In [118]:
import torch
from einops import rearrange, reduce, repeat

In [119]:
# Cross-entropy loss. Scalar value of the current cost. Approximated through batched sampling. Root node of backpropagaton pass -> non-convex optimization

def cross_entropy_loss(logits, targets) -> torch.float: # (... Vocab_size), (...)
    max_logit = reduce(logits, '... V -> ... 1', reduction='max')
    exp_norm_logits = torch.exp(logits - max_logit)
    sum_exps = reduce(exp_norm_logits, '... V -> ... 1', reduction='sum')
    gathered_logits = torch.gather(logits, dim=-1, index=repeat(targets, '... -> ... c', c=1)) # (...)
    log_probs = gathered_logits - max_logit - torch.log(sum_exps)
    return - ( reduce(log_probs, '... -> ', reduction='mean')) # averaged cost over batch and sequence

def cross_entropy_loss_logsumexp_fused_kernel(logits, targets): # 3 main kernels instead of 6: log + sum + exp + max get fused into ONE single kernel. Less HMB read/write -> higher arithmetic intensity operation
    gathered_logits = torch.gather(input=logits, dim=-1, index=targets.unsqueeze(-1) )
    batched_cross_entropy = - gathered_logits + torch.logsumexp(logits, dim=-1, keepdim=True) # (B T 1)
    return reduce(batched_cross_entropy, '... -> ', reduction='mean')

In [120]:
# Learning
# SGD with learning rate decreasing over time from Pytorch API

from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math

class SGD(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3):

        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")

        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue

                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or 0.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.

        return loss


weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=100)

for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.item())
    loss.backward()
    opt.step()


21.43465805053711
21.43465805053711
3.677605628967285
0.08801336586475372
8.719641709166486e-17
9.718581206037321e-19
3.272588551492895e-20
1.9495033470224887e-21
1.6724101101820224e-22
1.8582333330461063e-23


In [121]:
# Implement AdamW: (lr, weight_decay, beta1, beta2)
# Adam optimizer tracks a running the gradients moment of order 1 (hyperparameter: beta 1) and order 2 (beta 2)
# AdamW adds a weight decay at the optimizer .step() (instead of at the gradient level for example)

class AdamW(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3, weight_decay=0.01, betas=(0.9, 0.95), eps=1e-8):

        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")

        defaults = {"lr": lr, 'weight_decay': weight_decay, 'betas': betas, 'eps': eps}
        super().__init__(params, defaults)

    def step(self, closure = None):
        loss = None if closure is None else closure()

        for group in self.param_groups:
            lr, weight_decay, (beta1, beta2), eps = group["lr"], group["weight_decay"], group["betas"], group["eps"] # Get the hyperparameters
            for p in group["params"]:
                if p.grad is None:
                    continue

                state = self.state[p] # Get state associated with p
                if len(state) == 0:
                    state['moment_order_1'] = torch.zeros_like(p)
                    state['moment_order_2'] = torch.zeros_like(p)
                    state['t'] = 1

                t = state['t']
                grad = p.grad
                moment_order_1 = state['moment_order_1'] * beta1 + (1 - beta1) * grad
                moment_order_2 = state['moment_order_2'] * beta2 + (1 - beta2) * grad**2
                adjusted_lr = lr * math.sqrt(1 - beta2**t) / (1 - beta1**t)
                with torch.no_grad():
                    p -= p * weight_decay * lr + adjusted_lr * moment_order_1 / (torch.sqrt(moment_order_2) + eps) # AdamW weight update. -= makes the update in_place, very important!

                state['moment_order_1'] = moment_order_1 # stateful buffer of total size 4 * num_param (torch.float32)
                state['moment_order_2'] = moment_order_2 # stateful buffer of total size 4 * num_param (torch.float32)
                state["t"] = t + 1 

        return loss

weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = AdamW([weights], lr=100)

for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.item())
    loss.backward()
    opt.step()

20.236846923828125
9999.9248046875
5054.7060546875
68.57032012939453
8.544849395751953
1.3984979391098022
0.2649034857749939
0.05467662215232849
0.011872122064232826
0.0026502814143896103
